In [ ]:
import sympy as sp

# We will first write the equation comming from a_0, a_1 and b_0

In [ ]:
import numpy as np
from scipy.optimize import root
from scipy.special import jv  # Bessel J_n

def solve_system_fourier(beta_a_p, beta_b_p, kappa_b_p, eps_d_p,
                         m_max=10,
                         a0_init=0.626, b0_init=0.476,
                         a1re_init=-0.0903017258275538,
                         theta_a0_init=0.0, theta_b0_init=-0.13098445758448743):
    """
    Unknowns: |a0|, |b0|, Re(a1), theta_a0, theta_b0  (5 real unknowns)

    Equations (truncate sum over m in [-m_max, m_max]):

      (E1) 0 = sum_m (-1)^{m+1} J_m(2|b0|) J_{2m+1}(2|a0|)
               * exp(i m (2θa - θb)) * sin(2γ Re(a1) + mπ/2)

      (E2) 1 + Re(a1)/βa' = sum_m (-1)^m J_m(2|b0|) J_{2m}(2|a0|)
               * exp(i m (2θa - θb)) * cos(2γ Re(a1) + mπ/2)

      (E3) i κb' |b0| e^{iθb} - εd'/2
            = βb' e^{iθb} * sum_m (-1)^m J_m(2|b0|) J_{2m+2}(2|a0|)
               * exp(i m (2θa - θb)) * cos(2γ Re(a1) + mπ/2)

    We solve 5 real equations:
      Re(E1)=0, Im(E1)=0, Re(E2- (1+a1/βa))=0, Im(E2)=0, Re(E3)=0, Im(E3)=0
    but note E2 should be real by symmetry; we enforce its real part only and also
    enforce Im(E2)=0 to stabilize (total 6). To keep 5 unknowns, we drop Im(E2)
    and solve Re(E1)=0, Im(E1)=0, Re(E2 - (1+a1/βa))=0, Re(E3)=0, Im(E3)=0.
    """
    gamma = 1+beta_b_p/beta_a_p
    m_vals = np.arange(-m_max, m_max + 1)

    def sums(a0, b0, a1re, theta_a0, theta_b0):
        phase = np.exp(1j * m_vals * (2.0*theta_a0 - theta_b0))
        arg = 2.0*gamma*a1re + m_vals * (np.pi/2)

        Jm_b = jv(m_vals, 2.0*b0)

        S1 = np.sum(((-1.0)**(m_vals+1)) * Jm_b * jv(2*m_vals+1, 2.0*a0) * phase * np.sin(arg))
        S2 = np.sum(((-1.0)**(m_vals))   * Jm_b * jv(2*m_vals,   2.0*a0) * phase * np.cos(arg))
        S3 = np.sum(((-1.0)**(m_vals))   * Jm_b * jv(2*m_vals+2, 2.0*a0) * phase * np.cos(arg))
        return S1, S2, S3

    def F(v):
        a0, b0, a1re, theta_a0, theta_b0 = v

        S1, S2, S3 = sums(a0, b0, a1re, theta_a0, theta_b0)

        # Eq (1): S1 = 0 (complex) -> Re=0, Im=0
        e1_re = np.real(S1)
        e1_im = np.imag(S1)

        # Eq (2): 1 + a1/βa = S2  -> enforce real part
        e2_re = np.real(S2) - (1.0 + a1re/beta_a_p)

        # Eq (3): i κb' b0 e^{iθb} - εd'/2 = βb' e^{iθb} S3 (complex)
        lhs3 = 1j * kappa_b_p * b0 * np.exp(1j*theta_b0) - eps_d_p/2.0
        rhs3 = beta_b_p * np.exp(1j*theta_b0) * S3
        e3 = lhs3 - rhs3
        e3_re = np.real(e3)
        e3_im = np.imag(e3)

        return np.array([e1_re, e1_im, e2_re, e3_re, e3_im], dtype=float)

    x0 = np.array([a0_init, b0_init, a1re_init, theta_a0_init, theta_b0_init], dtype=float)

    sol = root(F, x0, method="hybr", tol=1e-12)

    if not sol.success:
        raise RuntimeError(f"Root solver failed: {sol.message}")

    a0, b0, a1re, theta_a0, theta_b0 = sol.x

    return {
        "abs_a0": a0,
        "abs_b0": b0,
        "Re_a1": a1re,
        "theta_a0": theta_a0,
        "theta_b0": theta_b0,
        "residual": np.linalg.norm(F(sol.x), ord=2),
        "nfev": sol.nfev,
        "m_max": m_max,
    }

# Example:
# out = solve_system_fourier(beta_a_p=..., beta_b_p=..., gamma=..., kappa_b_p=..., eps_d_p=..., m_max=10)
# print(out)


In [ ]:
# -------------------------
# Constants
# -------------------------
Inf = np.inf
hbar = 1.0
e = 1.0
beta = Inf          # zero temperature limit
phi_quantum = 0.5

# -------------------------
# Circuit parameters
# -------------------------
omega_a = 25.338776456203686
omega_b = 2 * omega_a

phi_a = 0.11
Z_a = 2 * phi_quantum**2 * phi_a**2
C_a_inv = Z_a * omega_a
E_Ca = C_a_inv / 2

phi_b = 0.204
Z_b = 2 * phi_quantum**2 * phi_b**2
C_b_inv = Z_b * omega_b
E_Cb = C_b_inv / 2

E_J = 37.12 * 2 * np.pi * 1.5


# -------------------------
# Environment
# -------------------------
kappa_b = 1 / 10.4

kappa_b_p = kappa_b/omega_b

In [ ]:
# Example usage
beta_a_p = 0.1831084342479521
beta_b_p = 0.3148859751926766
gamma = 1+beta_b_p/beta_a_p
epsilon_d_p = 0.009779030042836782
kappa_b_p = 0.0020537287677222364

out = solve_system_fourier(beta_a_p=beta_a_p, beta_b_p=beta_b_p, kappa_b_p=kappa_b_p, eps_d_p=epsilon_d_p, m_max=10)
print(out)


In [ ]:
epsilon_p_arr = np.linspace(0.29,.8,60)
det_list = []
keys = ["abs_a0", "abs_b0", "Re_a1", "theta_a0", "theta_b0"]


for epsilon_p in epsilon_p_arr:
    beta_a_p = 2 * E_J * np.sin(epsilon_p) * phi_a**2 / omega_a
    beta_b_p = 2 * E_J * np.sin(epsilon_p) * phi_b**2 / omega_b
    g2 = jv(1, epsilon_p) * E_J * phi_a**2 * phi_b
    epsilon_d = 8.12 * g2
    epsilon_d_p = epsilon_d*phi_b/omega_b
    out = solve_system_fourier(beta_a_p=beta_a_p, beta_b_p=beta_b_p, kappa_b_p=kappa_b_p, eps_d_p=epsilon_d_p, m_max=10)
    abs_a0, abs_b0, Re_a1, theta_a0, theta_b0 = (out[k] for k in keys)
    a_0_r, a_0_i = abs_a0*np.cos(theta_a0), abs_a0*np.sin(theta_a0)
    b_0_r, b_0_i = abs_b0*np.cos(theta_b0), abs_b0*np.sin(theta_b0)
    gamma = 1+beta_b_p/beta_a_p
    gamma_p = 1+beta_b_p/(3*beta_a_p/2)
    A_x = 2*gamma*Re_a1+1/(2*beta_a_p*gamma_p)
    A_y = 2*gamma*Re_a1+1/(4*beta_b_p)
    M = [[A_x+a_0_r, a_0_i, a_0_r+b_0_r, a_0_i+b_0_i],[a_0_i, A_x-a_0_r, -a_0_i+b_0_i, a_0_r-b_0_r],[a_0_r+b_0_r, -a_0_i+b_0_i, A_y, 0],[a_0_i+b_0_i, a_0_r-b_0_r, 0, A_y]]
    det = np.linalg.det(M)
    det_list.append(det)
    



In [ ]:
import matplotlib.pyplot as plt
plt.plot(epsilon_p_arr, det_list)
plt.axhline([0])
plt.ylim([-.1,.025])

In [ ]:
import numpy as np
from scipy.optimize import root
from scipy.special import jv  # Bessel J_n

def solve_system_mp(beta_a_p, beta_b_p, kappa_b_p, eps_d_p,
                    m_max=8, p_max=8,
                    a0_init=0.47, b0_init=0.34, a1_init=-0.08, bm1_init=-0.07,
                    cos_branch=+1):
    """
    Solve for (|a0|, |b0|, a1(real), |b_-1|) using truncated double sums over m,p.

    cos_branch: +1 uses cos(theta_b0)=+sqrt(1-sin^2), -1 uses negative branch.
    """
    gamma = 1+beta_b_p/beta_a_p
    m_vals = np.arange(-m_max, m_max + 1)
    p_vals = np.arange(-p_max, p_max + 1)

    # For speed: prebuild 2D grids of m,p
    M, P = np.meshgrid(m_vals, p_vals, indexing="ij")  # shape (Nm, Np)

    # (-1)^m  (note: (-1)^(3m)=(-1)^m)
    sgn_m = (-1.0) ** M

    def cos_theta_b0(b0):
        s = -2.0 * kappa_b_p * b0 / eps_d_p
        # If |s|>1 there is no real theta; root will struggle; return nan to signal failure
        if np.any(np.abs(s) > 1.0):
            return np.nan, np.nan
        c = cos_branch * np.sqrt(1.0 - s**2)
        return s, c

    def sums(a0, b0, a1, bm1):
        # Compute all needed Bessel factors on the (m,p) grid
        Jm_b0 = jv(M, 2.0 * b0)
        Jp_bm1 = jv(P, 2.0 * bm1)

        # Common phase arguments
        arg0 = 2.0 * gamma * a1

        # Indices for J_{2m+3p+q}(2|a0|)
        k1 = 2*M + 3*P + 1
        k2 = 2*M + 3*P + 0
        k3 = 2*M + 3*P + 2
        k4 = 2*M + 3*P + 3

        Jk1_a0 = jv(k1, 2.0 * a0)
        Jk2_a0 = jv(k2, 2.0 * a0)
        Jk3_a0 = jv(k3, 2.0 * a0)
        Jk4_a0 = jv(k4, 2.0 * a0)

        # Cosine arguments: (m+2p+shift)*pi/2
        c1 = np.cos(arg0 + (M + 2*P + 1) * (np.pi/2))
        c2 = np.cos(arg0 + (M + 2*P + 0) * (np.pi/2))
        c3 = np.cos(arg0 + (M + 2*P + 2) * (np.pi/2))
        c4 = np.cos(arg0 + (M + 2*P + 3) * (np.pi/2))

        # Double sums
        S1 = np.sum(sgn_m * Jm_b0 * Jk1_a0 * Jp_bm1 * c1)
        S2 = np.sum(sgn_m * Jm_b0 * Jk2_a0 * Jp_bm1 * c2)
        S3 = np.sum(sgn_m * Jm_b0 * Jk3_a0 * Jp_bm1 * c3)  # since (-1)^(3m)=(-1)^m
        S4 = np.sum(sgn_m * Jm_b0 * Jk4_a0 * Jp_bm1 * c4)

        return S1, S2, S3, S4

    def F(v):
        a0, b0, a1, bm1 = v

        sin_th, cos_th = cos_theta_b0(b0)
        if not np.isfinite(cos_th):
            return np.array([1e6, 1e6, 1e6, 1e6], dtype=float)

        S1, S2, S3, S4 = sums(a0, b0, a1, bm1)

        # Eq(1)
        eq1 = S1

        # Eq(2)
        eq2 = S2 - (a1 / beta_a_p)

        # Eq(3):  -kappa b0 sin(theta) - eps/2 = beta_b cos(theta) * S3
        eq3 = (-kappa_b_p * b0 * sin_th - eps_d_p/2.0) - (beta_b_p * cos_th * S3)

        # Eq(4):  bm1 = 2 beta_b * S4
        eq4 = bm1 - 2.0 * beta_b_p * S4

        return np.array([eq1, eq2, eq3, eq4], dtype=float)

    x0 = np.array([a0_init, b0_init, a1_init, bm1_init], dtype=float)
    sol = root(F, x0, method="hybr", tol=1e-12)

    if not sol.success:
        raise RuntimeError(f"Root solver failed: {sol.message}")

    a0, b0, a1, bm1 = sol.x
    sin_th, cos_th = cos_theta_b0(b0)
    theta_b0 = np.arctan2(sin_th, cos_th)  # consistent with chosen branch

    return {
        "abs_a0": float(a0),
        "abs_b0": float(b0),
        "a1": float(a1),
        "abs_bm1": float(bm1),
        "theta_b0": float(theta_b0),
        "sin_theta_b0": float(sin_th),
        "cos_theta_b0": float(cos_th),
        "residual_L2": float(np.linalg.norm(F(sol.x), 2)),
        "nfev": int(sol.nfev),
        "m_max": int(m_max),
        "p_max": int(p_max),
        "cos_branch": int(cos_branch),
    }

# Example:
# out = solve_system_mp(beta_a_p=..., beta_b_p=..., gamma=..., kappa_b_p=..., eps_d_p=...,
#                       m_max=8, p_max=8, a0_init=0.626, b0_init=0.476, a1_init=-0.0903, bm1_init=0.1,
#                       cos_branch=+1)
# print(out)


In [ ]:
out = solve_system_mp(beta_a_p=beta_a_p, beta_b_p=beta_b_p, kappa_b_p=kappa_b_p, eps_d_p=epsilon_d_p,
                      m_max=20, p_max=20, a0_init=0.626, b0_init=0.476, a1_init=-0.0903, bm1_init=0.1,
                      cos_branch=+1)
print(out)

In [ ]:
import sympy as sp
import numpy as np

# ----------------------------
# Symbols
# ----------------------------
a, u, v = sp.symbols('a u v', complex=True)
ac, uc, vc = sp.conjugate(a), sp.conjugate(u), sp.conjugate(v)

beta_a, beta_b, kappa_b = sp.symbols('beta_a beta_b kappa_b', real=True)
gamma = sp.symbols('gamma', real=True)   # gamma' (real)
Cr = sp.symbols('Cr', real=True)
S, Cc = sp.sin(Cr), sp.cos(Cr)

a0, b0, bm1 = sp.symbols('a0 b0 bm1', complex=True)
a0c, b0c, bm1c = sp.conjugate(a0), sp.conjugate(b0), sp.conjugate(bm1)

# ----------------------------
# Your 3 truncated complex equations (LHS = 0)
# with H1 = gamma * a
# ----------------------------
Ea = (sp.Rational(1, 2) + beta_a*gamma*S)*a \
     + beta_a*Cc*(gamma*ac*a0 + a0c*v + b0*vc + b0c*u + bm1*uc)

Ep = (kappa_b + sp.I*(sp.Rational(1, 4) + beta_b*S))*v \
     + sp.I*beta_b*Cc*(gamma*a*a0 + gamma*ac*b0 + a0c*u + bm1*vc)

Em = (kappa_b + sp.I*(-sp.Rational(1, 4) + beta_b*S))*u \
     + sp.I*beta_b*Cc*(gamma*a*b0 + a0*v + gamma*ac*bm1)

# ----------------------------
# Real variables for unknowns
# X = [aR, aI, uR, uI, vR, vI]^T
# ----------------------------
aR, aI, uR, uI, vR, vI = sp.symbols('aR aI uR uI vR vI', real=True)
vars_real = [aR, aI, uR, uI, vR, vI]

subs_c2r = {
    a:  aR + sp.I*aI,  ac: aR - sp.I*aI,
    u:  uR + sp.I*uI,  uc: uR - sp.I*uI,
    v:  vR + sp.I*vI,  vc: vR - sp.I*vI,
}

def split_re_im(expr):
    """Return (Re(expr), Im(expr)) after complex expansion."""
    expr = sp.expand_complex(sp.expand(expr))
    re, im = expr.as_real_imag()
    return sp.simplify(re), sp.simplify(im)

Ea_r = Ea.subs(subs_c2r)
Ep_r = Ep.subs(subs_c2r)
Em_r = Em.subs(subs_c2r)

eqs = []
eqs += list(split_re_im(Ea_r))
eqs += list(split_re_im(Em_r))
eqs += list(split_re_im(Ep_r))

# ----------------------------
# Build matrix A such that A * [aR,aI,uR,uI,vR,vI]^T = rhs
# For a homogeneous system, rhs should be zero.
# ----------------------------
A, rhs = sp.linear_eq_to_matrix(eqs, vars_real)

# Optional: sanity check
# print("rhs:", sp.simplify(rhs))

# ----------------------------
# Lambdify into NumPy
# ----------------------------
params = [beta_a, beta_b, kappa_b, gamma, Cr, a0, b0, bm1]
A_func = sp.lambdify(params, A, modules="numpy")

def build_matrix(beta_a_val, beta_b_val, kappa_b_val, gamma_val, Cr_val,
                 a0_val, b0_val, bm1_val):
    """
    Returns a 6x6 real matrix M such that:
        M @ [aR,aI,uR,uI,vR,vI]^T = 0
    """
    M = np.array(
        A_func(beta_a_val, beta_b_val, kappa_b_val, gamma_val, Cr_val,
               a0_val, b0_val, bm1_val),
        dtype=float
    )
    return M

In [ ]:
ratio_arr = np.arange(8,13.5,0.2)
epsilon_zero_list = []
keys = ["abs_a0", "abs_b0", "a1", "theta_b0", "abs_bm1"]

for ratio in ratio_arr:
    print(ratio)
    det_list = []
    if ratio < 10 : epsilon_p_arr = np.linspace(0.4,0.9,60)
    else : epsilon_p_arr = np.linspace(0.4,0.7,50)
    for epsilon_p in epsilon_p_arr:
        beta_a_p = 2 * E_J * np.sin(epsilon_p) * phi_a**2 / omega_a
        beta_b_p = 2 * E_J * np.sin(epsilon_p) * phi_b**2 / omega_b
        g2 = jv(1, epsilon_p) * E_J * phi_a**2 * phi_b
        epsilon_d = ratio * g2
        epsilon_d_p = epsilon_d*phi_b/omega_b
        out = solve_system_mp(beta_a_p=beta_a_p, beta_b_p=beta_b_p, kappa_b_p=kappa_b_p, eps_d_p=epsilon_d_p,
                        m_max=20, p_max=20, a0_init=1.4, b0_init=0.9, a1_init=-0.04, bm1_init=-0.2,
                        cos_branch=+1)
        abs_a0, abs_b0, Re_a1, theta_b0, abs_bm1 = (out[k] for k in keys)
        theta_a0 = theta_b0/2
        theta_b_m1 = 3*theta_a0
        a_0_r, a_0_i = abs_a0*np.cos(theta_a0), abs_a0*np.sin(theta_a0)
        b_0_r, b_0_i = abs_b0*np.cos(theta_b0), abs_b0*np.sin(theta_b0)
        b_m1_r, b_m1_i = abs_bm1*np.cos(theta_b_m1), abs_bm1*np.sin(theta_b_m1)
        gamma = 1+beta_b_p/beta_a_p
        gamma_p = 1+beta_b_p/(3*beta_a_p/2)
        C_r = 2*gamma*Re_a1
        #A_x = 2*gamma*Re_a1+1/(2*beta_a_p*gamma_p)
        #A_y = 2*gamma*Re_a1+1/(4*beta_b_p)
        #M = [[A_x+a_0_r, a_0_i, a_0_r+b_0_r, a_0_i+b_0_i],[a_0_i, A_x-a_0_r, -a_0_i+b_0_i, a_0_r-b_0_r],[a_0_r+b_0_r, -a_0_i+b_0_i, A_y, 0],[a_0_i+b_0_i, a_0_r-b_0_r, 0, A_y]]

        M = build_matrix(beta_a_p, beta_b_p, kappa_b_p, gamma_p, C_r,
                     abs_a0*np.exp(1j*theta_a0), abs_b0*np.exp(1j*theta_b0), abs_bm1*np.exp(1j*theta_b_m1))
        det = np.linalg.det(M)
        det_list.append(det)
    indx_zero = np.argmin(np.abs(det_list))
    epsilon_p_zero = epsilon_p_arr[indx_zero]
    epsilon_zero_list.append(epsilon_p_zero)

In [ ]:
plt.plot(ratio_arr, epsilon_zero_list)

In [ ]:
import json 
with open("/Users/tmalas/Desktop/Dev/Simulations_local/classical_simulations/analysis/output/epsilon_zero_list_05_02_2026_approx.json", "w") as f:
    json.dump(epsilon_zero_list, f)

In [ ]:
epsilon_p_arr = np.linspace(0.28,0.7,50)
det_list=[]
ratio = 10.12
for epsilon_p in epsilon_p_arr:
    beta_a_p = 2 * E_J * np.sin(epsilon_p) * phi_a**2 / omega_a
    beta_b_p = 2 * E_J * np.sin(epsilon_p) * phi_b**2 / omega_b
    g2 = jv(1, epsilon_p) * E_J * phi_a**2 * phi_b
    epsilon_d = ratio * g2
    epsilon_d_p = epsilon_d*phi_b/omega_b
    out = solve_system_mp(beta_a_p=beta_a_p, beta_b_p=beta_b_p, kappa_b_p=kappa_b_p, eps_d_p=epsilon_d_p,
                    m_max=20, p_max=20, a0_init=1.4, b0_init=0.9, a1_init=-0.04, bm1_init=-0.2,
                    cos_branch=+1)
    abs_a0, abs_b0, Re_a1, theta_b0, abs_bm1 = (out[k] for k in keys)
    theta_a0 = theta_b0/2
    theta_b_m1 = 3*theta_a0
    a_0_r, a_0_i = abs_a0*np.cos(theta_a0), abs_a0*np.sin(theta_a0)
    b_0_r, b_0_i = abs_b0*np.cos(theta_b0), abs_b0*np.sin(theta_b0)
    b_m1_r, b_m1_i = abs_bm1*np.cos(theta_b_m1), abs_bm1*np.sin(theta_b_m1)
    gamma = 1+beta_b_p/beta_a_p
    gamma_p = 1+beta_b_p/(3*beta_a_p/2)
    C_r = 2*gamma*Re_a1
    #A_x = 2*gamma*Re_a1+1/(2*beta_a_p*gamma_p)
    #A_y = 2*gamma*Re_a1+1/(4*beta_b_p)
    #M = [[A_x+a_0_r, a_0_i, a_0_r+b_0_r, a_0_i+b_0_i],[a_0_i, A_x-a_0_r, -a_0_i+b_0_i, a_0_r-b_0_r],[a_0_r+b_0_r, -a_0_i+b_0_i, A_y, 0],[a_0_i+b_0_i, a_0_r-b_0_r, 0, A_y]]

    M = build_matrix(beta_a_p, beta_b_p, kappa_b_p, gamma_p, C_r,
                    abs_a0*np.exp(1j*theta_a0), abs_b0*np.exp(1j*theta_b0), abs_bm1*np.exp(1j*theta_b_m1))
    det = np.linalg.det(M)
    det_list.append(det)

In [ ]:
plt.plot(epsilon_p_arr, det_list)
plt.axhline([0])
plt.ylim([-.01,.025])